In [14]:

import pandas as pd
import numpy as np
import gc

from data.dataset import MalwareDatasetLoader
from data.data_processing import split_out_targets, force_dense, preprocess_existing, preprocess_fit

RELOAD_DATA = False
if not RELOAD_DATA:
  try:
    print(df_features_train.head())
  except Exception as e:
    print("No dataframe.  Loading data...")
    RELOAD_DATA=True
if RELOAD_DATA:
  df_loader = MalwareDatasetLoader()

  df_train, df_val, df_test = df_loader.make_data_splits()
  
  df_features_train, df_y_train = split_out_targets(df_train)
  df_features_val, df_y_val = split_out_targets(df_val)
  df_features_test, df_y_test = split_out_targets(df_test)
  del df_loader
  gc.collect()


   id.orig_p  id.resp_p  duration  orig_bytes  resp_bytes  missed_bytes  \
0    60836.0       23.0  3.145479         0.0         0.0           0.0   
1     8341.0    62336.0 -1.000000        -1.0        -1.0           0.0   
2    58186.0       23.0 -1.000000        -1.0        -1.0           0.0   
3    52808.0       23.0  3.122224         0.0         0.0           0.0   
4    50196.0       23.0  0.000001         0.0         0.0           0.0   

   orig_pkts  orig_ip_bytes  resp_pkts  resp_ip_bytes proto service conn_state  
0        6.0          360.0        0.0            0.0   tcp       -         S0  
1        0.0            0.0        0.0            0.0   tcp       -        OTH  
2        1.0           60.0        0.0            0.0   tcp       -         S0  
3        3.0          180.0        0.0            0.0   tcp       -         S0  
4        2.0          120.0        0.0            0.0   tcp       -         S0  


In [15]:
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix


def compute_metrics(classifier, df_features, df_y):
  print(f"Compute Metrics Start: {df_features.shape[0]}")
  predictions = classifier.predict(df_features)
  y_prob = classifier.predict_proba(df_features)[:, 1]
  print("Compute Metrics End")

  acc = accuracy_score(df_y, predictions)
  f1 = f1_score(df_y, predictions)
  auc = roc_auc_score(df_y, y_prob)
  cm = confusion_matrix(df_y, predictions)

  print(f"Accuracy: {acc:.4f}")
  print(f"F1:       {f1:.4f}")
  print(f"AUC:      {auc:.4f}")
  print("Confusion matrix:")
  print(cm)
  print()
  return classification_report(df_y, predictions)


In [ ]:
RANDOM_STATE=2025

from data.data_processing import preprocess_fit

import sklearn

def train_bagging(df_features_train, df_y_train, max_depth=2, max_trees=50, max_samples=0.5, bootstrap=False):
  tree_classifier = sklearn.tree.DecisionTreeClassifier(max_depth=max_depth)

  bagging_classifier = sklearn.ensemble.BaggingClassifier(
    estimator=tree_classifier,
    n_estimators=max_trees,
    max_samples=max_samples,
    bootstrap=bootstrap,
    n_jobs=32,
    random_state=RANDOM_STATE)

  bagging_classifier.fit(df_features_train, df_y_train)

  return bagging_classifier

print("Preprocessing Start")
X_transformed, preprocessor = preprocess_fit(df_features_train, quantile_clipping=False)
print("Preprocessing Done")
for (max_depth, max_trees, max_samples, bootstrap) in [
    (40, 500, 0.01, False),
    (50, 3, 0.5, False),
    (2, 500, 0.5, False),
    (10, 200, 0.2, False),
    (10, 200, 0.5, True),
    (100, 10, 0.1, False),
    (100, 1, 0.8, False),
    (500, 1, 1.0, False),
    (500, 3, 0.8, False),
    (1000, 1, 1.0, False),
 ]:
  print("Training bagging model")
  print(f"Max Depth: {max_depth}")
  print(f"Max Trees: {max_trees}")
  print(f"Sample Ratio: {max_samples}")
  print(f"Bootstrap: {bootstrap}")

  bagging_classifier = train_bagging(X_transformed, df_y_train,
                                      max_depth=max_depth,
                                      max_trees=max_trees,
                                      max_samples=max_samples,
                                      bootstrap=bootstrap)
  X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

  metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
  print(metrics)

  X_test_transformed = preprocess_existing(df_features_test, preprocessor)

  metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
  print(metrics_test)

Preprocessing Start
Global feature space dimensionality: 33
Preprocessing Done
Training bagging model
Max Depth: 40
Max Trees: 500
Sample Ratio: 0.01
Bootstrap: False


/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.9928
F1:       0.9898
AUC:      0.9985
Confusion matrix:
[[2408551   26691]
 [    376 1316032]]

              precision    recall  f1-score   support

           0       1.00      0.99      0.99   2435242
           1       0.98      1.00      0.99   1316408

    accuracy                           0.99   3751650
   macro avg       0.99      0.99      0.99   3751650
weighted avg       0.99      0.99      0.99   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.9928
F1:       0.9898
AUC:      0.9985
Confusion matrix:
[[2407890   26703]
 [    368 1316690]]

              precision    recall  f1-score   support

           0       1.00      0.99      0.99   2434593
           1       0.98      1.00      0.99   1317058

    accuracy                           0.99   3751651
   macro avg       0.99      0.99      0.99   3751651
weighted avg       0.99      0.99      0.99   3751651

Training bagging model
Max

/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.9931
F1:       0.9903
AUC:      0.9981
Confusion matrix:
[[2409624   25618]
 [    144 1316264]]

              precision    recall  f1-score   support

           0       1.00      0.99      0.99   2435242
           1       0.98      1.00      0.99   1316408

    accuracy                           0.99   3751650
   macro avg       0.99      0.99      0.99   3751650
weighted avg       0.99      0.99      0.99   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.9931
F1:       0.9903
AUC:      0.9982
Confusion matrix:
[[2408958   25635]
 [    149 1316909]]

              precision    recall  f1-score   support

           0       1.00      0.99      0.99   2434593
           1       0.98      1.00      0.99   1317058

    accuracy                           0.99   3751651
   macro avg       0.99      0.99      0.99   3751651
weighted avg       0.99      0.99      0.99   3751651

Training bagging model
Max

/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.8643
F1:       0.8378
AUC:      0.9001
Confusion matrix:
[[1928092  507150]
 [   1886 1314522]]

              precision    recall  f1-score   support

           0       1.00      0.79      0.88   2435242
           1       0.72      1.00      0.84   1316408

    accuracy                           0.86   3751650
   macro avg       0.86      0.90      0.86   3751650
weighted avg       0.90      0.86      0.87   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.8641
F1:       0.8377
AUC:      0.8999
Confusion matrix:
[[1926810  507783]
 [   1899 1315159]]

              precision    recall  f1-score   support

           0       1.00      0.79      0.88   2434593
           1       0.72      1.00      0.84   1317058

    accuracy                           0.86   3751651
   macro avg       0.86      0.89      0.86   3751651
weighted avg       0.90      0.86      0.87   3751651

Training bagging model
Max

/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


KeyboardInterrupt: 

In [23]:
for (max_depth, max_trees, max_samples, bootstrap) in [
    (10000, 1, 1.0, False),
    (500, 10, 0.3, False),
 ]:
  print("Training bagging model")
  print(f"Max Depth: {max_depth}")
  print(f"Max Trees: {max_trees}")
  print(f"Sample Ratio: {max_samples}")
  print(f"Bootstrap: {bootstrap}")

  bagging_classifier = train_bagging(X_transformed, df_y_train,
                                      max_depth=max_depth,
                                      max_trees=max_trees,
                                      max_samples=max_samples,
                                      bootstrap=bootstrap)
  X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

  metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
  print(metrics)

  X_test_transformed = preprocess_existing(df_features_test, preprocessor)

  metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
  print(metrics_test)

Training bagging model
Max Depth: 10000
Max Trees: 1
Sample Ratio: 1.0
Bootstrap: False


/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.9963
F1:       0.9947
AUC:      0.9987
Confusion matrix:
[[2421354   13888]
 [    168 1316240]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2435242
           1       0.99      1.00      0.99   1316408

    accuracy                           1.00   3751650
   macro avg       0.99      1.00      1.00   3751650
weighted avg       1.00      1.00      1.00   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.9963
F1:       0.9948
AUC:      0.9987
Confusion matrix:
[[2420932   13661]
 [    175 1316883]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2434593
           1       0.99      1.00      0.99   1317058

    accuracy                           1.00   3751651
   macro avg       0.99      1.00      1.00   3751651
weighted avg       1.00      1.00      1.00   3751651

Training bagging model
Max

/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.9959
F1:       0.9942
AUC:      0.9988
Confusion matrix:
[[2420061   15181]
 [    168 1316240]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2435242
           1       0.99      1.00      0.99   1316408

    accuracy                           1.00   3751650
   macro avg       0.99      1.00      1.00   3751650
weighted avg       1.00      1.00      1.00   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.9960
F1:       0.9943
AUC:      0.9988
Confusion matrix:
[[2419593   15000]
 [    152 1316906]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2434593
           1       0.99      1.00      0.99   1317058

    accuracy                           1.00   3751651
   macro avg       0.99      1.00      1.00   3751651
weighted avg       1.00      1.00      1.00   3751651



In [22]:
for (max_depth, max_trees, max_samples, bootstrap) in [
    #(500, 1, 1.0, False),
    (500, 3, 0.8, False),
    (1000, 1, 1.0, False),
 ]:
  print("Training bagging model")
  print(f"Max Depth: {max_depth}")
  print(f"Max Trees: {max_trees}")
  print(f"Sample Ratio: {max_samples}")
  print(f"Bootstrap: {bootstrap}")

  bagging_classifier = train_bagging(X_transformed, df_y_train,
                                      max_depth=max_depth,
                                      max_trees=max_trees,
                                      max_samples=max_samples,
                                      bootstrap=bootstrap)
  X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

  metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
  print(metrics)

  X_test_transformed = preprocess_existing(df_features_test, preprocessor)

  metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
  print(metrics_test)

Training bagging model
Max Depth: 500
Max Trees: 3
Sample Ratio: 0.8
Bootstrap: False


/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.9962
F1:       0.9947
AUC:      0.9988
Confusion matrix:
[[2421254   13988]
 [    163 1316245]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2435242
           1       0.99      1.00      0.99   1316408

    accuracy                           1.00   3751650
   macro avg       0.99      1.00      1.00   3751650
weighted avg       1.00      1.00      1.00   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.9963
F1:       0.9947
AUC:      0.9988
Confusion matrix:
[[2420840   13753]
 [    158 1316900]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2434593
           1       0.99      1.00      0.99   1317058

    accuracy                           1.00   3751651
   macro avg       0.99      1.00      1.00   3751651
weighted avg       1.00      1.00      1.00   3751651

Training bagging model
Max

/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:930: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.9963
F1:       0.9947
AUC:      0.9987
Confusion matrix:
[[2421354   13888]
 [    168 1316240]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2435242
           1       0.99      1.00      0.99   1316408

    accuracy                           1.00   3751650
   macro avg       0.99      1.00      1.00   3751650
weighted avg       1.00      1.00      1.00   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.9963
F1:       0.9948
AUC:      0.9987
Confusion matrix:
[[2420932   13661]
 [    175 1316883]]

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   2434593
           1       0.99      1.00      0.99   1317058

    accuracy                           1.00   3751651
   macro avg       0.99      1.00      1.00   3751651
weighted avg       1.00      1.00      1.00   3751651



Training bagging model
Global feature space dimensionality: 33
Preprocessing Done


TypeError: train_bagging() got an unexpected keyword argument 'max_samples'